# demo_harness

> Phase 7 communication harness: four channels on the benchmark world, evaporation, circuit teams, range check and budget scaling.

Spec: `specs/multiagent/phase7/` (design-low-level section 8). Artifacts under `results/multiagent/demo_harness/{quick,full}/` (gitignored): PNGs, `metrics.json`, and **`report.md`**. Quick: 300 s, seeds 0–2; `#| slow`: 600 s (1200 s circuit), seeds 0–9, harness check and budget scaling.

In [ ]:
from explRL.circuit_train import historyFromRecords, piScore
from explRL.harness_runs import (HarnessConfig, benchWorld, circuitWorld, netHarvest, runHarness, rangeCheck)
from explRL.team_refs import idealFree, ifdDistance, ledgerArrays, occupancy, socialOptimum, timeToIFD
from explRL.train import save_demo_figure, save_demo_metrics, save_demo_report
import matplotlib.pyplot as plt
import numpy as np

DN = 'multiagent/demo_harness'
CHANNELS = ('none', 'blackboard', 'messages', 'hierarchical')
CIRCUIT_CHANNELS = ('none', 'blackboard', 'messages')
artifact_mode, T, circuit_T = 'quick', 300.0, 300.0
SEEDS = list(range(3))
N_BANDITS = 8
N_CIRCUIT = 4
DEFAULTS = HarnessConfig().asDict()
_bw = benchWorld()
WORLD = dict(layout=_bw.layout.name, L=_bw.L, travelScale=float(_bw.layout.T[0, 1] / _bw.layout.dist[0, 1]),
             model=_bw.model, nPatches=_bw.nPatches, nDanger=_bw.nDanger, intake=_bw.intake.tolist(), c=_bw.c,
             Tmove=_bw.Tmove, noise=_bw.noise, dangerCost=_bw.dangerCost)


def boot_ci(vals, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    vals = np.asarray(vals, float)
    boots = [rng.choice(vals, len(vals), replace=True).mean() for _ in range(n)]
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def cfg_channel(name, **kw):
    d = {**HarnessConfig(channel=name).asDict(), 'budget': 'time', **kw}
    return HarnessConfig.fromDict(d)


def bandit_series(cfg, N, seed, Ttot):
    c = HarnessConfig.fromDict({**cfg.asDict(), 'budget': 'time', 'T': Ttot})
    world, hist, channel, metrics = runHarness(c, N, seed)
    L = ledgerArrays(world.ledger)
    times, counts = occupancy(L, N, world.nPatches, Ttot)
    n_ifd, _ = idealFree(world, N)
    d = ifdDistance(counts, n_ifd, N)
    moves = [k * world.Tmove for k in range(1, int(Ttot // world.Tmove) + 1)
             if k * world.Tmove < Ttot] if world.Tmove else []
    bins = np.arange(0.0, Ttot + 10.0, 10.0)
    rate = np.histogram(L['t'], bins=bins, weights=netHarvest(world, L))[0] / 10.0   # welfare: harvest minus danger cost
    return dict(bins=bins[:-1], rate=rate, times=times, ifd_d=d, moves=moves, metrics=metrics)


def mean_series(runs, key):
    return np.mean([r[key] for r in runs], axis=0)


def summarize_bandit(cfg, N, seeds, Ttot):
    runs = [bandit_series(cfg, N, sd, Ttot) for sd in seeds]
    mets = [r['metrics'] for r in runs]
    pas = [m['perAgentSecond'] for m in mets]
    comm = [sum(m['comm'].get('perAgentSecond', {}).values()) for m in mets]
    tifd = []
    for m in mets:
        ok = [t for t in (m.get('timeToIFD') or []) if t == t]
        tifd.append(float(np.mean(ok)) if ok else float('nan'))
    ok_tifd = [x for x in tifd if x == x]
    return dict(
        perAgentSecond=dict(mean=float(np.mean(pas)), ci=boot_ci(pas)),
        commPerAgentSecond=dict(mean=float(np.mean(comm)), ci=boot_ci(comm)),
        timeToIFD=dict(mean=float(np.nanmean(tifd)) if ok_tifd else float('nan'),
                       ci=boot_ci(ok_tifd) if len(ok_tifd) > 1 else ((ok_tifd[0], ok_tifd[0]) if ok_tifd else (float('nan'), float('nan')))),
        series=dict(bins=runs[0]['bins'], rate=mean_series(runs, 'rate'), times=runs[0]['times'],
                    ifd_d=mean_series(runs, 'ifd_d'), moves=runs[0]['moves']),
    )


In [ ]:
#| slow
artifact_mode, T, circuit_T = 'full', 600.0, 1200.0
SEEDS = list(range(10))


## Channels (8 bandits, benchmark world)

In [ ]:
channels = {ch: summarize_bandit(cfg_channel(ch), N_BANDITS, SEEDS, T) for ch in CHANNELS}
ref_w = benchWorld(); ref_w.reset(SEEDS[0], N_BANDITS); _, y_so = socialOptimum(ref_w, N_BANDITS)
moves = channels['none']['series']['moves']

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ch in CHANNELS:
    s = channels[ch]['series']
    axes[0].step(s['bins'], s['rate'], where='post', label=ch)
    axes[1].plot(s['times'], s['ifd_d'], label=ch)
axes[0].axhline(y_so, color='k', ls=':', label='SO yield')
axes[0].set_ylabel('team welfare / s'); axes[0].legend(fontsize=8)
axes[1].set_ylabel('IFD distance'); axes[1].set_xlabel('time (s)')
for ax in axes:
    for mt in moves:
        ax.axvline(mt, color='0.5', lw=0.8, ls='--')
plt.tight_layout()
save_demo_figure('channels', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()
channels_export = {ch: {k: v for k, v in channels[ch].items() if k != 'series'} for ch in CHANNELS}


## Evaporation (blackboard, λ = 0, 0.01, 0.1)

In [ ]:
EVAP_LAM = (0.0, 0.01, 0.1)
evaporation = {str(lam): summarize_bandit(cfg_channel('blackboard', lam=lam), N_BANDITS, SEEDS, T)
               for lam in EVAP_LAM}

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for lam in EVAP_LAM:
    s = evaporation[str(lam)]['series']
    lab = f'λ={lam}'
    axes[0].step(s['bins'], s['rate'], where='post', label=lab)
    axes[1].plot(s['times'], s['ifd_d'], label=lab)
axes[0].set_ylabel('team harvest / s'); axes[0].legend(fontsize=8)
axes[1].set_ylabel('IFD distance'); axes[1].set_xlabel('time (s)')
for ax in axes:
    for mt in moves:
        ax.axvline(mt, color='0.5', lw=0.8, ls='--')
plt.tight_layout()
save_demo_figure('evaporation', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()
evaporation_export = {k: {kk: vv for kk, vv in v.items() if kk != 'series'} for k, v in evaporation.items()}


## Circuit channels (4 agents, crowded ring)

In [ ]:
circuit = {}
fig, axes = plt.subplots(1, len(CIRCUIT_CHANNELS), figsize=(3.2 * len(CIRCUIT_CHANNELS), 3.5), sharey='row')
if len(CIRCUIT_CHANNELS) == 1:
    axes = [axes]
for ax, ch in zip(axes, CIRCUIT_CHANNELS):
    cfg = HarnessConfig.fromDict({**HarnessConfig(channel=ch, agent='circuit').asDict(),
                                  'budget': 'time', 'T': circuit_T})
    world, hist, channel, metrics = runHarness(cfg, N_CIRCUIT, 0)
    Lc = ledgerArrays(world.ledger); netc = netHarvest(world, Lc)
    harvest = [float(netc[Lc['agent'] == i].sum()) for i in range(N_CIRCUIT)]   # net of danger cost
    pis = [float(piScore(historyFromRecords(hist.forAgent(i)))) for i in range(N_CIRCUIT)]
    circuit[ch] = dict(harvest=harvest, PI=pis)
    x = np.arange(N_CIRCUIT)
    w = 0.35
    ax.bar(x - w / 2, harvest, width=w, label='harvest')
    ax.bar(x + w / 2, pis, width=w, label='PI')
    ax.set_title(ch); ax.set_xlabel('agent'); ax.set_xticks(x)
axes[0].set_ylabel('harvest / PI')
axes[-1].legend(fontsize=8)
fig.suptitle('CircuitCommMember teams (asymmetric zone world)')
plt.tight_layout()
save_demo_figure('circuit_channels', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()
circuit_export = dict(scene_symmetric=False, channels=circuit)


## Harness check (AC-11, full run only)

In [ ]:
#| slow
RC_CFGS = {ch: HarnessConfig(channel=ch, agent='bandit') for ch in CHANNELS}
range_check = rangeCheck(RC_CFGS, T=T)
sizes = (1, 4, 16, 64)
fig, ax = plt.subplots(figsize=(8, 4))
xs = np.array(sizes)
for ch in CHANNELS:
    ys = [range_check['table'][str(N)][ch][0] for N in sizes]
    ax.plot(xs, ys, 'o-', label=ch)
for N in sizes:
    ax.plot(N, range_check['table'][str(N)]['random'][0], 'kx', ms=8)
ax.set_xscale('log', base=2); ax.set_xlabel('team size'); ax.set_ylabel('team harvest / s (whole run)')
ax.set_title(f"range check: ceiling={range_check['ceiling']} floor={range_check['floor']} gap={range_check['gap']} ok={range_check['ok']}")
ax.legend(fontsize=8)
plt.tight_layout()
save_demo_figure('range_check', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()


## Budget scaling (full run only)

In [ ]:
#| slow
BUDGET_SIZES = (1, 2, 4, 8, 16, 32, 64)
BUDGET_SEEDS = list(range(5))
budgets = {}
for budget in ('time', 'compute'):
    budgets[budget] = {}
    for ch in CHANNELS:
        row = {}
        for N in BUDGET_SIZES:
            pas = []
            for sd in BUDGET_SEEDS:
                kw = dict(channel=ch, agent='bandit', budget=budget, T=T)
                if budget == 'compute':
                    kw['B'] = DEFAULTS['B']
                cfg = HarnessConfig.fromDict({**HarnessConfig().asDict(), **kw})
                pas.append(runHarness(cfg, N, sd)[3]['perAgentSecond'])
            row[str(N)] = dict(mean=float(np.mean(pas)), ci=boot_ci(pas, seed=2))
        budgets[budget][ch] = row

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
xs = np.array(BUDGET_SIZES)
for ax, budget in zip(axes, ('time', 'compute')):
    for ch in CHANNELS:
        m = np.array([budgets[budget][ch][str(x)]['mean'] for x in xs])
        lo, hi = np.array([budgets[budget][ch][str(x)]['ci'] for x in xs]).T
        ax.errorbar(xs, m, yerr=[m - lo, hi - m], capsize=3, label=ch)
    ax.set_xscale('log', base=2); ax.set_ylabel('welfare / agent-s')
    ax.set_title(f'fixed {budget} budget'); ax.legend(fontsize=7, ncol=2)
axes[-1].set_xlabel('team size')
plt.tight_layout()
save_demo_figure('scaling_budgets', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()


## Metrics export

In [ ]:
metrics = dict(mode=artifact_mode, seeds=SEEDS, world=WORLD, defaults=DEFAULTS,
               channels=channels_export, evaporation=evaporation_export, circuit=circuit_export,
               T_s=T, circuit_T_s=circuit_T, nBandits=N_BANDITS)
if artifact_mode == 'full':
    metrics['range_check'] = range_check
    metrics['budgets'] = budgets


def harness_demo_report(m):
    lines = [
        f"# Phase 7 harness demo ({m['mode']} run)\n",
        f"Benchmark ring world ({m['world']['nPatches']} patches, danger sites, patches jump every {m['world']['Tmove']} s). "
        f"Eight bandit learners compare four communication channels; circuit teams use four trained agents on a crowded asymmetric ring. "
        f"Bandit runs: {m['T_s']} s world time; circuit: {m['circuit_T_s']} s.\n",
        "## Figures in this folder\n",
        "- `channels.png` — team harvest rate (10 s bins, mean over seeds) and IFD distance; dashed lines = patch moves.\n",
        "- `evaporation.png` — blackboard with λ = 0, 0.01, 0.1 (same panels as channels).\n",
        "- `circuit_channels.png` — harvest and performance index per agent for none / blackboard / messages.\n",
    ]
    if m['mode'] == 'full':
        lines += [
            "- `range_check.png` — AC-11 dynamic range (welfare per agent-s vs team size).\n",
            "- `scaling_budgets.png` — welfare per agent-s vs team size under fixed time and fixed compute budgets.\n",
        ]
    lines += ["- `metrics.json` — numbers behind this report.\n", "\nWelfare = true harvest minus the danger cost (time at danger sites x `dangerCost`).\n", "## Channel comparison\n"]
    for ch in CHANNELS:
        row = m['channels'][ch]
        pas = row['perAgentSecond']
        comm = row['commPerAgentSecond']
        tifd = row['timeToIFD']
        lines.append(
            f"- **{ch}**: {pas['mean']:.3f} welfare per agent-s (95% CI {pas['ci'][0]:.3f}–{pas['ci'][1]:.3f}); "
            f"comm cost {comm['mean']:.4f}/agent-s; mean time-to-IFD after moves {tifd['mean']:.0f} s (when defined).\n"
        )
    lines.append("\n## Evaporation\n")
    for lam, row in m['evaporation'].items():
        pas = row['perAgentSecond']['mean']
        lines.append(f"- Blackboard λ={lam}: {pas:.3f} welfare per agent-s.\n")
    lines.append("\n## Circuit teams\n")
    for ch, row in m['circuit']['channels'].items():
        h = ', '.join(f'{x:.0f}' for x in row['harvest'])
        lines.append(f"- **{ch}**: welfare totals [{h}]; PI {[round(p, 2) for p in row['PI']]}.\n")
    if 'range_check' in m:
        rc = m['range_check']
        lines += [
            "\n## Harness check (AC-11)\n",
            f"- Ceiling (first-quarter rate < 0.9× social optimum): {'pass' if rc['ceiling'] else 'fail'}.\n",
            f"- Floor (none channel above random): {'pass' if rc['floor'] else 'fail'}.\n",
            f"- Channel gap (paired best−worst): {'pass' if rc['gap'] else 'fail'}.\n",
            f"- Overall: {'OK' if rc['ok'] else 'needs benchWorld tuning'}.\n",
        ]
    if 'budgets' in m:
        lines.append("\n## Budget scaling\n")
        for budget in ('time', 'compute'):
            best = max(CHANNELS, key=lambda ch: m['budgets'][budget][ch]['64']['mean'])
            val = m['budgets'][budget][best]['64']['mean']
            lines.append(f"- Fixed **{budget}** at 64 agents: highest mean welfare/agent-s is **{best}** ({val:.3f}).\n")
    lines.append("\nRe-run: `nbdev-test --n-workers 0 nbs/demo_harness.ipynb` (quick) or add `--flags slow` (full). Spec: `specs/multiagent/phase7/`.\n")
    return ''.join(lines)


save_demo_metrics(metrics, mode=artifact_mode, demo_name=DN)
rpath = save_demo_report(harness_demo_report(metrics), mode=artifact_mode, demo_name=DN)
print('wrote', rpath.name)
print('none perAgentSecond:', channels_export['none']['perAgentSecond'])
